# Review PII Candidates

Cursor-based triage: ноутбук показывает батч кандидатов, ты указываешь только хорошие индексы через `accept([...])`, затем `finish_batch()` двигает курсор. Хорошие примеры сразу пишутся в clean JSONL/CSV.


In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path.cwd()
while PROJECT_DIR.name and not (PROJECT_DIR / 'pyproject.toml').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / 'real_pii_collection' / 'data'
CANDIDATES_DIR = DATA_DIR / 'outputs' / 'llm_candidates'
REVIEW_DIR = DATA_DIR / 'outputs' / 'reviewed_candidates'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_FILE_NAME = None  # None = первый *.candidates.jsonl по алфавиту

files = sorted(CANDIDATES_DIR.glob('*.candidates.jsonl'))
if CANDIDATE_FILE_NAME is None:
    if not files:
        raise FileNotFoundError(f'No *.candidates.jsonl in {CANDIDATES_DIR}')
    candidates_path = files[0]
else:
    candidates_path = CANDIDATES_DIR / CANDIDATE_FILE_NAME
    if not candidates_path.exists():
        raise FileNotFoundError(candidates_path)

review_state_path = REVIEW_DIR / f'{candidates_path.stem}.review_state.json'
clean_jsonl_path = REVIEW_DIR / f'{candidates_path.stem}.clean.jsonl'
clean_csv_path = REVIEW_DIR / f'{candidates_path.stem}.clean.csv'

print('candidates:', candidates_path)
print('review_state:', review_state_path)
print('clean_jsonl:', clean_jsonl_path)
print('available:', [p.name for p in files])

candidates: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/llm_candidates/dvach_posts.candidates.jsonl
review_state: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.review_state.json
clean_jsonl: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.jsonl
available: ['dvach_posts.candidates.jsonl', 'dvach_posts_keywords.candidates.jsonl']


In [7]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def load_review_state(path: Path) -> dict:
    if not path.exists():
        return {'last_reviewed_idx': -1, 'accepted_indices': []}
    state = json.loads(path.read_text(encoding='utf-8'))
    # Backward compatibility with older cursor-based state.
    if 'last_reviewed_idx' not in state:
        state['last_reviewed_idx'] = int(state.get('cursor', 0)) - 1
    state.setdefault('accepted_indices', [])
    return state


def save_review_state() -> None:
    tmp = review_state_path.with_suffix(review_state_path.suffix + '.tmp')
    tmp.write_text(json.dumps(review_state, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(review_state_path)


def write_clean_outputs() -> None:
    accepted = sorted(set(review_state.get('accepted_indices', [])))
    clean_rows = [rows[idx] for idx in accepted if 0 <= idx < len(rows)]
    with clean_jsonl_path.open('w', encoding='utf-8') as f:
        for r in clean_rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    df = pd.DataFrame(clean_rows)
    if len(df):
        flat = df.copy()
        if 'llm_pii_types' in flat:
            flat['llm_pii_types'] = flat['llm_pii_types'].map(lambda x: ', '.join(x) if isinstance(x, list) else x)
        flat.to_csv(clean_csv_path, index=False)
    elif clean_csv_path.exists():
        clean_csv_path.unlink()


rows = load_jsonl(candidates_path)
review_state = load_review_state(review_state_path)
accepted_set = set(review_state.get('accepted_indices', []))

print('total candidates:', len(rows))
print('last_reviewed_idx:', review_state.get('last_reviewed_idx', -1))
print('next_idx:', int(review_state.get('last_reviewed_idx', -1)) + 1)
print('accepted:', len(accepted_set))

total candidates: 3328
last_reviewed_idx: 3327
next_idx: 3328
accepted: 568


In [3]:
BATCH_SIZE = 30


def print_candidate(idx: int) -> None:
    r = rows[idx]
    text = r.get('text') or ''
    print('=' * 120)
    print(f"IDX {idx} | risk={r.get('llm_risk')} | board={r.get('board')} | post={r.get('post_num')} | len={r.get('text_len')}")
    print('types:', r.get('llm_pii_types'))
    print('comment:', r.get('llm_comment', ''))
    print(text)


def parse_indices(raw: str, start: int, end: int) -> list[int]:
    raw = raw.strip()
    if not raw:
        return []
    raw = raw.replace(',', ' ')
    indices = []
    for part in raw.split():
        if '-' in part:
            left, right = part.split('-', 1)
            a, b = int(left), int(right)
            if a > b:
                a, b = b, a
            indices.extend(range(a, b + 1))
        else:
            indices.append(int(part))
    bad = [idx for idx in indices if idx < start or idx >= end]
    if bad:
        raise ValueError(f'Индексы вне текущего батча {start}-{end - 1}: {bad}')
    return sorted(set(indices))


def review_loop(batch_size: int = BATCH_SIZE):
    global review_state
    review_state = load_review_state(review_state_path)

    while True:
        start = int(review_state.get('last_reviewed_idx', -1)) + 1
        if start >= len(rows):
            print('review complete')
            break

        end = min(start + batch_size, len(rows))
        print('\n' + '#' * 120)
        print(f'BATCH {start}-{end - 1} / {len(rows) - 1} | accepted={len(set(review_state.get("accepted_indices", [])))}')
        print('#' * 120)
        for idx in range(start, end):
            print_candidate(idx)

        print('=' * 120)
        print('Введи индексы, которые надо СОХРАНИТЬ, например: 12 15 18-20')
        print('Enter = сохранить ничего и перейти дальше; q = выйти без сохранения прогресса текущего батча')
        raw = input('keep indices> ').strip()

        if raw.lower() in {'q', 'quit', 'exit'}:
            print('stopped; last_reviewed_idx unchanged:', review_state.get('last_reviewed_idx', -1))
            save_review_state()
            break

        if raw == "":
            indices = []
            print("no accepted indices; marking batch as reviewed")
        else:
            indices = parse_indices(raw, start, end)
        accepted = set(review_state.get('accepted_indices', []))
        accepted.update(indices)
        review_state['accepted_indices'] = sorted(accepted)
        review_state['last_reviewed_idx'] = end - 1
        save_review_state()
        write_clean_outputs()

        print(f'accepted this batch: {indices}')
        print(f'last_reviewed_idx={end - 1}/{len(rows) - 1} | next_idx={end} | total_accepted={len(accepted)}')

    write_clean_outputs()
    print('clean_jsonl:', clean_jsonl_path)
    print('clean_csv:', clean_csv_path)


# Запуск:
# review_loop()

In [5]:
## Запусти эту ячейку для интерактивного отсмотра.
# На каждом батче вводи только индексы хороших кандидатов.
review_loop()


########################################################################################################################
BATCH 2529-2558 / 3327 | accepted=445
########################################################################################################################
IDX 2529 | risk=high | board=pr | post=3469811 | len=773
types: ['ссылка на личный профиль', 'резюме']
comment: Сообщение содержит ссылку на личный профиль (easyoffer.ru) и упоминание о резюме, что может быть связано с конкретным человеком.
>>3469576

https://easyoffer.ru/analytic/android_developer

Сделал все отклики через этот сайт на хх ру, результат 1 собеседоване даже не на разработчика, а на тестировшика, вообщем на хх ру все мертво, буду регать линкед ин на днях(
ну и попробую к уже готовому резюме ещё два года накрутить хули терять то нечего а в пятидесятый раз покупать новую симку впадлу)
, так щас проект делаю на заказ.
Что первый раз тут ? 
Я ищу работу уже год, я сделал 1 к + откликов (ну это в цел

keep indices>  2531 2554 


accepted this batch: [2531, 2554]
last_reviewed_idx=2558/3327 | next_idx=2559 | total_accepted=447

########################################################################################################################
BATCH 2559-2588 / 3327 | accepted=447
########################################################################################################################
IDX 2559 | risk=high | board=pr | post=3649449 | len=3407
types: ['ссылка на профиль']
comment: Содержит ссылку на профиль пользователя GitHub (Даниэль Стенберг) в контексте обсуждения проблем с AI.
@monkey, есть кого обоссать тут?

--- 

GitHub действительно рассматривает отключение PR — правда.

Product manager Camilla Moraes 28 января 2026 открыла официальное обсуждение на GitHub Community. Среди рассматриваемых мер: полное отключение PR, ограничение только для коллабораторов, удаление PR из интерфейса, атрибуция AI-использования.
Источник: [GitHub Community Discussion #185387](
https://github.com/orgs/communi

keep indices>  2560 2580 2585


accepted this batch: [2560, 2580, 2585]
last_reviewed_idx=2588/3327 | next_idx=2589 | total_accepted=450

########################################################################################################################
BATCH 2589-2618 / 3327 | accepted=450
########################################################################################################################
IDX 2589 | risk=medium | board=pr | post=3427022 | len=813
types: ['место работы', 'организация']
comment: Упоминается завод по производству пива и сливок, что может быть связано с конкретной организацией.
>>3426984

>Про RS-232 помню про TX, RX, GND, CTS, RTS. А вот названия и подключение остальных модемных сигналов не вспомню без гугла.

Ну а я даже не ебу что это. Я просто знаю что жто какой-то устаревшей COM порт, типо того. Если что-то общее можешь рассказать то не обоссут наверное. Меня же конкретные вещи не спрашивали. Меня просто спросили, что я знаю про физический уровень. Я хз что сказать. Во всех

...

accepted this batch: [3249, 3268, 3277, 3278]
last_reviewed_idx=3278/3327 | next_idx=3279 | total_accepted=553

########################################################################################################################
BATCH 3279-3308 / 3327 | accepted=553
########################################################################################################################
IDX 3279 | risk=medium | board=soc | post=7323485 | len=230
types: ['location', 'personal_story']
comment: Сообщение содержит упоминание района (Коломна) и личные детали, которые могут помочь идентифицировать личность
>>7294787

Я снимаю коммуналку за 12к комнату, мне норм, только метро нет рядом, район Коломна, тут так дёшево потому что видимо одни алкаши, старичьё и долбоебы тут живут, кстати, кто с коломны пишите, можем погулять вдоль пряжки
IDX 3280 | risk=high | board=soc | post=7327922 | len=32
types: ['phone_number']
comment: Сообщение содержит телефонный номер: +905548717555
>>7327921

телефон

keep indices>  3279 3280 3281 3286 3288 3308 


accepted this batch: [3279, 3280, 3281, 3286, 3288, 3308]
last_reviewed_idx=3308/3327 | next_idx=3309 | total_accepted=559

########################################################################################################################
BATCH 3309-3327 / 3327 | accepted=559
########################################################################################################################
IDX 3309 | risk=high | board=un | post=1084504 | len=245
types: ['ФИО', 'место работы', 'город']
comment: Упоминается одноклассница, её профессия, город (Дубай) и факт работы там, что может быть связано с конкретным человеком.
>>1084502

У нас тоже есть университет спорта с ебанутой аббревиатурой, моя одноклассница его окончила. Я бы приложил её фотки, но они гуглятся. Так что наслово верь что там красивая накаченная девушка. Сейчас она в Дубаях физкультуру преподаёт.
IDX 3310 | risk=high | board=un | post=1084578 | len=774
types: ['ФИО (косвенно, через описание личного опыта)', 'место раб

keep indices>  3311 3312 3315 3316 3317 3318 3319 3321 3325 


accepted this batch: [3311, 3312, 3315, 3316, 3317, 3318, 3319, 3321, 3325]
last_reviewed_idx=3327/3327 | next_idx=3328 | total_accepted=568
review complete
clean_jsonl: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.jsonl
clean_csv: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.csv


In [6]:
def review_summary():
    state = load_review_state(review_state_path)
    accepted = set(state.get('accepted_indices', []))
    last_idx = int(state.get('last_reviewed_idx', -1))
    next_idx = last_idx + 1
    print('total:', len(rows))
    print('last_reviewed_idx:', last_idx, f"({next_idx / max(len(rows), 1):.1%} done)")
    print('next_idx:', next_idx)
    print('accepted:', len(accepted))
    print('clean_jsonl:', clean_jsonl_path)
    print('clean_csv:', clean_csv_path)

review_summary()

total: 3328
last_reviewed_idx: 3327 (100.0% done)
next_idx: 3328
accepted: 568
clean_jsonl: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.jsonl
clean_csv: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.csv


### Convert to LabelStudio

In [25]:
import json

checked_texts = []
with open(str(clean_jsonl_path), 'r') as f:
    for line in f:
        text = json.loads(line)
        checked_texts.append(text['text'])
        
label_studio_tasks = [{'data':{'text':t}} for t in checked_texts]

with open(str(DATA_DIR)+'/outputs/' + 'collected_real_pii_texts.json', 'w') as f:
    json.dump(label_studio_tasks,f)

In [6]:
str(clean_jsonl_path)

'/Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/reviewed_candidates/dvach_posts.candidates.clean.jsonl'

### Convert from LabelStudio

In [12]:
import json
from genson import SchemaBuilder

with open(str(DATA_DIR)+'/outputs/' + 'annotated_label_studio.json', 'r') as f:
    annotated = json.load(f)

builder = SchemaBuilder()
builder.add_object(annotated[0])

schema = builder.to_schema()

print(json.dumps(schema, ensure_ascii=False, indent = 2))

{
  "$schema": "http://json-schema.org/schema#",
  "type": "object",
  "properties": {
    "id": {
      "type": "integer"
    },
    "annotations": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "id": {
            "type": "integer"
          },
          "completed_by": {
            "type": "integer"
          },
          "result": {
            "type": "array",
            "items": {
              "type": "object",
              "properties": {
                "value": {
                  "type": "object",
                  "properties": {
                    "start": {
                      "type": "integer"
                    },
                    "end": {
                      "type": "integer"
                    },
                    "text": {
                      "type": "string"
                    },
                    "labels": {
                      "type": "array",
                      "items": {
            

In [18]:
annotated[3]

{'id': 3480,
 'annotations': [{'id': 1143,
   'completed_by': 1,
   'result': [{'value': {'start': 0,
      'end': 15,
      'text': 'ЗАО ЭнергоСтрой',
      'labels': ['ORGANIZATION']},
     'id': 'Fji-WRYHoF',
     'from_name': 'label',
     'to_name': 'text',
     'type': 'labels',
     'origin': 'manual'},
    {'value': {'start': 16,
      'end': 60,
      'text': 'город Екатеринбург,ул.Начдива Васильева д.3а',
      'labels': ['ADDRESS']},
     'id': 'cduN9QcvWK',
     'from_name': 'label',
     'to_name': 'text',
     'type': 'labels',
     'origin': 'manual'}],
   'was_cancelled': False,
   'ground_truth': False,
   'created_at': '2026-05-10T20:27:49.322192Z',
   'updated_at': '2026-05-10T20:27:49.322207Z',
   'draft_created_at': '2026-05-10T20:27:39.522598Z',
   'lead_time': 20.361,
   'prediction': {},
   'result_count': 2,
   'unique_id': '1b1ccd2a-7730-4384-bc0b-4ae27c380a03',
   'import_id': None,
   'last_action': None,
   'bulk_created': False,
   'task': 3480,
   'projec

In [19]:
annotated[3]['data']

{'text': 'ЗАО ЭнергоСтрой,город Екатеринбург,ул.Начдива Васильева д.3а,о нем я писал. Что касается долгов и арестов,это нужно индивидуально решать с бухгалтерией,они подскажут варианты решения проблемы.Прикрепленные фото это очередь на вахтовку. На некоторых объектах такие очереди везде,в столовку,в сортир,в душ,на работу и обратно.К этому нужно быть готовым.'}

In [31]:
annotated[4]['annotations'][0]['result']

[{'value': {'start': 111,
   'end': 128,
   'text': 'КРАСНОЯРСКИЙ КРАЙ',
   'labels': ['ADDRESS']},
  'id': 'oXs632Yopn',
  'from_name': 'label',
  'to_name': 'text',
  'type': 'labels',
  'origin': 'manual'},
 {'value': {'start': 499,
   'end': 514,
   'text': 'РЕСПУБЛИКА ТЫВА',
   'labels': ['ADDRESS']},
  'id': 'HeBk6iqO5y',
  'from_name': 'label',
  'to_name': 'text',
  'type': 'labels',
  'origin': 'manual'},
 {'value': {'start': 991,
   'end': 1007,
   'text': 'ХАБАРОВСКИЙ КРАЙ',
   'labels': ['ADDRESS']},
  'id': 'HypHIbe576',
  'from_name': 'label',
  'to_name': 'text',
  'type': 'labels',
  'origin': 'manual'},
 {'value': {'start': 1200,
   'end': 1221,
   'text': 'ЛЕНИНГРАДСКАЯ ОБЛАСТЬ',
   'labels': ['ADDRESS']},
  'id': 'KUSYNCcr3z',
  'from_name': 'label',
  'to_name': 'text',
  'type': 'labels',
  'origin': 'manual'},
 {'value': {'start': 1695,
   'end': 1711,
   'text': '+7 996-428-03-53',
   'labels': ['PHONE_NUMBER']},
  'id': 'rStb2X-Az7',
  'from_name': 'label',
  't

In [ ]:
entities_counter = dict()
converted_data = []
conversion_errors = []

for task_idx, annotation in enumerate(annotated):
    text = annotation['data']['text']
    entities_list = []

    annotations = annotation.get('annotations') or []
    if not annotations:
        converted_data.append({'text': text, 'entities': entities_list})
        continue

    for entity_idx, entity_dict in enumerate(annotations[0].get('result', [])):
        value = entity_dict['value']
        start = value['start']
        end = value['end']
        entity_text = value['text']
        label = value['labels'][0]

        extracted = text[start:end]
        if extracted != entity_text:
            conversion_errors.append({
                'task_idx': task_idx,
                'entity_idx': entity_idx,
                'start': start,
                'end': end,
                'label': label,
                'expected_text': entity_text,
                'actual_text': extracted,
            })

        entity = {
            'start': start,
            'end': end,
            'label': label,
            'text': entity_text,
        }

        entities_counter[label] = entities_counter.get(label, 0) + 1
        entities_list.append(entity)

    entities_list.sort(key=lambda item: (item['start'], item['end'], item['label']))
    converted_data.append({'text': text, 'entities': entities_list})

print(f'converted texts: {len(converted_data)}')
print(f'converted entities: {sum(len(item["entities"]) for item in converted_data)}')
print(f'conversion errors: {len(conversion_errors)}')


In [69]:
labels_df = pd.DataFrame(entities_counter.values(), index = entities_counter.keys()).sort_values(by = 0,ascending = False)

In [70]:
labels_df

,0
ADDRESS,219
AGE,173
NAME,144
TELEGRAM,122
NICKNAME,112
ORGANIZATION,109
SEX,69
EDUCATION,65
EMAIL,34
HEALTH,29


In [74]:
output_jsonl_path = DATA_DIR / 'outputs' / 'converted_real_pii_texts.jsonl'

with output_jsonl_path.open('w', encoding='utf-8') as f:
    for item in converted_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print('saved:', output_jsonl_path)


saved: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/converted_real_pii_texts.jsonl
